In [1]:
 """Session 29: pretrained ResNet-34 U-Net, fixed grouped CV on the V3 coarse export.
Run the accompanying self-contained Kaggle notebook. CLI --help lists overrides.
No torchvision, MMDetection, segmentation-models-pytorch, custom CUDA, or dataset modifications.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import random
import socket
import subprocess
import sys
import time
import zipfile

CLASSES = ['background', 'Nail_psoriasis', 'SJS-TEN', 'acne', 'vitiligo']
BUILD = 'SESSION29-UNET-R34-V1'


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False) + '\n')
    tmp.replace(path)


def find_data(explicit=None):
    if explicit:
        roots = [Path(explicit).expanduser().resolve()]
    else:
        roots = []
        if Path('/kaggle/input').is_dir():
            roots.extend(p.parent.parent for p in Path('/kaggle/input').rglob('metadata/manifest.csv'))
            roots.extend(extract_attached_v3_archives(Path('/kaggle/input')))
        cache = Path('/kaggle/working/session29_dataset_cache')
        if cache.is_dir():
            roots.extend(p.parent.parent for p in cache.rglob('metadata/manifest.csv'))
        local = Path(__file__).resolve().parent / 'rebuilt_datasets/skin_dataset_v3_coarse'
        if local.is_dir():
            roots.append(local)
        roots = sorted(set(p.resolve() for p in roots if (p/'reports/v3_completion.json').is_file()))
    if len(roots) != 1:
        raise ValueError(f'Expected one V3 dataset, found {roots}. Set --data-root explicitly.')
    root = roots[0]
    with (root/'metadata/manifest.csv').open() as f:
        rows = list(csv.DictReader(f))
    validate_manifest(root, rows)
    return root, rows


def extract_attached_v3_archives(input_root):
    """Safely extract a V3 ZIP attached by Kaggle, only when needed.

    Kaggle can expose an uploaded dataset either as files or as its original
    ZIP. Inputs stay read-only: extraction occurs in /kaggle/working and is
    reused by parent and child fold processes.
    """
    if not Path('/kaggle/working').is_dir():
        return []
    found = []
    archives = sorted(p for p in input_root.rglob('*.zip') if 'skin_dataset_v3_coarse' in p.name.lower())
    for archive in archives:
        digest = hashlib.sha256(archive.read_bytes()).hexdigest()[:16]
        destination = Path('/kaggle/working/session29_dataset_cache')/digest
        root = destination/'skin_dataset_v3_coarse'
        marker = root/'reports/v3_completion.json'
        if marker.is_file():
            found.append(root)
            continue
        with zipfile.ZipFile(archive) as zipped:
            invalid = zipped.testzip()
            if invalid:
                raise ValueError(f'Corrupt attached dataset ZIP member: {invalid}')
            members = zipped.infolist()
            if not any(item.filename == 'skin_dataset_v3_coarse/reports/v3_completion.json' for item in members):
                continue
            for item in members:
                name = Path(item.filename)
                target = (destination/name).resolve()
                if name.is_absolute() or '..' in name.parts or not target.is_relative_to(destination.resolve()):
                    raise ValueError(f'Unsafe ZIP member: {item.filename}')
                # ZIP symlinks are not dataset files and are rejected explicitly.
                if (item.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError(f'Symlink ZIP member is not allowed: {item.filename}')
            destination.mkdir(parents=True, exist_ok=True)
            for item in members:
                zipped.extract(item, destination)
        if not marker.is_file():
            raise RuntimeError(f'Attached ZIP extracted without required V3 marker: {archive}')
        print(f'Extracted attached V3 dataset ZIP to {root}', flush=True)
        found.append(root)
    return found


def validate_manifest(root, rows):
    if not rows:
        raise ValueError('Empty dataset')
    ids, groups, pixels = set(), {}, {}
    for r in rows:
        if r['image_id'] in ids:
            raise ValueError('Duplicate image ID')
        ids.add(r['image_id'])
        if r['split'] not in ('train', 'valid', 'test'):
            raise ValueError('Unknown split')
        partition = 'test' if r['split'] == 'test' else 'fold_' + r['cv_fold']
        if r['split'] != 'test' and r['cv_fold'] not in ('0', '1', '2', '3', '4'):
            raise ValueError('Invalid CV fold')
        for registry, key in ((groups, r['split_group_id']), (pixels, r['pixel_sha256'])):
            if not key or (key in registry and registry[key] != partition):
                raise ValueError(f'Group/pixel leakage: {key}')
            registry[key] = partition
        for key in ('image_file', 'semantic_mask_file'):
            p = (root/r[key]).resolve()
            if not p.is_relative_to(root.resolve()) or not p.is_file():
                raise ValueError(f'Missing/unsafe dataset path: {p}')
    for fold in range(5):
        for valid in (True, False):
            selected = [r for r in rows if r['split'] != 'test' and ((int(r['cv_fold']) == fold) == valid)]
            present = set()
            for r in selected:
                present.update(r['classes'].split('|'))
            if not selected or not set(CLASSES[1:]).issubset(present):
                raise ValueError(f'Fold {fold} has missing classes: {present}')


def metrics_from_confusion(cm):
    import numpy as np
    cm = np.asarray(cm, dtype=np.float64)
    tp = cm.diagonal()
    gt, pred = cm.sum(1), cm.sum(0)
    dice_den, iou_den = gt + pred, gt + pred - tp
    dice = np.divide(2*tp, dice_den, out=np.full(5, np.nan), where=dice_den > 0)
    iou = np.divide(tp, iou_den, out=np.full(5, np.nan), where=iou_den > 0)
    clean = lambda values: [None if not np.isfinite(v) else float(v) for v in values]
    return {'foreground_macro_dice': float(np.nanmean(dice[1:])),
            'foreground_macro_iou': float(np.nanmean(iou[1:])),
            'per_class_dice': dict(zip(CLASSES, clean(dice))),
            'per_class_iou': dict(zip(CLASSES, clean(iou))),
            'confusion_matrix': cm.astype('int64').tolist(),
            'aggregation': 'pooled native-resolution pixels; macro foreground classes; absent-both class undefined'}


def load_ml():
    # Lazy imports keep CLI/data validation independent of GPU libraries.
    global torch, nn, F, np, Image, ImageEnhance
    import numpy as np
    from PIL import Image, ImageEnhance
    import torch
    from torch import nn
    from torch.nn import functional as F


def make_resnet34(pretrained):
    # Standard torchvision-compatible ResNet-34 topology and state-dict names.
    # Reference: https://docs.pytorch.org/vision/main/_modules/torchvision/models/resnet.html
    class BasicBlock(nn.Module):
        def __init__(self, cin, cout, stride=1):
            super().__init__()
            self.conv1=nn.Conv2d(cin,cout,3,stride=stride,padding=1,bias=False)
            self.bn1=nn.BatchNorm2d(cout)
            self.relu=nn.ReLU(inplace=True)
            self.conv2=nn.Conv2d(cout,cout,3,padding=1,bias=False)
            self.bn2=nn.BatchNorm2d(cout)
            self.downsample=(nn.Sequential(nn.Conv2d(cin,cout,1,stride=stride,bias=False),
                                           nn.BatchNorm2d(cout)) if stride!=1 or cin!=cout else None)
        def forward(self,x):
            identity=x if self.downsample is None else self.downsample(x)
            z=self.relu(self.bn1(self.conv1(x)))
            return self.relu(self.bn2(self.conv2(z))+identity)
    class ResNet34(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1=nn.Conv2d(3,64,7,stride=2,padding=3,bias=False)
            self.bn1=nn.BatchNorm2d(64);self.relu=nn.ReLU(inplace=True)
            self.maxpool=nn.MaxPool2d(3,stride=2,padding=1)
            self.layer1=nn.Sequential(BasicBlock(64,64),*[BasicBlock(64,64) for _ in range(2)])
            self.layer2=nn.Sequential(BasicBlock(64,128,2),*[BasicBlock(128,128) for _ in range(3)])
            self.layer3=nn.Sequential(BasicBlock(128,256,2),*[BasicBlock(256,256) for _ in range(5)])
            self.layer4=nn.Sequential(BasicBlock(256,512,2),*[BasicBlock(512,512) for _ in range(2)])
            self.avgpool=nn.AdaptiveAvgPool2d((1,1)); self.fc=nn.Linear(512,1000)
            for m in self.modules():
                if isinstance(m,nn.Conv2d):nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
                elif isinstance(m,nn.BatchNorm2d):nn.init.ones_(m.weight);nn.init.zeros_(m.bias)
    base=ResNet34()
    assert sum(p.numel() for p in base.parameters())==21797672
    if pretrained:
        old_timeout=socket.getdefaulttimeout()
        socket.setdefaulttimeout(60)
        try:
            weights=torch.hub.load_state_dict_from_url(
                'https://download.pytorch.org/models/resnet34-b627a593.pth',
                map_location='cpu',check_hash=True,weights_only=True)
        finally:
            socket.setdefaulttimeout(old_timeout)
        base.load_state_dict(weights,strict=True)
    return base


def make_model(pretrained=True):
    class Block(nn.Module):
        def __init__(self, cin, cout):
            super().__init__()
            self.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.GroupNorm(8, cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                nn.GroupNorm(8, cout), nn.ReLU(inplace=True))
        def forward(self, x, skip=None):
            if skip is not None:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
                x = torch.cat([x, skip], 1)
            return self.net(x)
    class UNet(nn.Module):
        def __init__(self):
            super().__init__()
            base = make_resnet34(pretrained)
            self.encoder = nn.ModuleDict({'stem': nn.Sequential(base.conv1, base.bn1, base.relu),
                'pool': base.maxpool, 'l1': base.layer1, 'l2': base.layer2,
                'l3': base.layer3, 'l4': base.layer4})
            self.d3 = Block(512+256, 256)
            self.d2 = Block(256+128, 128)
            self.d1 = Block(128+64, 64)
            self.d0 = Block(64+64, 32)
            self.final = Block(32, 32)
            self.head = nn.Conv2d(32, 5, 1)
        def forward(self, x):
            size = x.shape[-2:]
            a = self.encoder['stem'](x)
            b = self.encoder['l1'](self.encoder['pool'](a))
            c = self.encoder['l2'](b)
            d = self.encoder['l3'](c)
            e = self.encoder['l4'](d)
            z = self.d0(self.d1(self.d2(self.d3(e, d), c), b), a)
            z = F.interpolate(z, size=size, mode='bilinear', align_corners=False)
            return self.head(self.final(z))
    return UNet()


def prepare_image(image, size):
    # Aspect-preserving letterbox; matching mask padding is ignored by the loss.
    w, h = image.size
    scale = size/max(w, h)
    nw, nh = max(1, round(w*scale)), max(1, round(h*scale))
    x, y = (size-nw)//2, (size-nh)//2
    canvas = Image.new('RGB', (size, size), (124, 116, 104))
    canvas.paste(image.resize((nw, nh), Image.Resampling.BILINEAR), (x, y))
    arr = np.asarray(canvas, dtype=np.float32).copy()/255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1)
    tensor = (tensor-torch.tensor([.485, .456, .406])[:, None, None])/torch.tensor([.229, .224, .225])[:, None, None]
    return tensor, (x, y, nw, nh)


class SkinDataset:
    def __init__(self, root, rows, size, train):
        self.root, self.rows, self.size, self.train = root, rows, size, train
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        if 'torch' not in globals():load_ml()
        r = self.rows[idx]
        with Image.open(self.root/r['image_file']) as src:
            image = src.convert('RGB')
        with Image.open(self.root/r['semantic_mask_file']) as src:
            mask = src.copy()
        if self.train:
            if random.random() < .5:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            angle = random.uniform(-15, 15)
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=(124,116,104))
            mask = mask.rotate(angle, resample=Image.Resampling.NEAREST, fillcolor=255)
            image = ImageEnhance.Brightness(image).enhance(random.uniform(.9, 1.1))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(.9, 1.1))
        tensor, (x,y,nw,nh) = prepare_image(image, self.size)
        padded = Image.new('L', (self.size,self.size), 255)
        padded.paste(mask.resize((nw,nh), Image.Resampling.NEAREST), (x,y))
        return tensor, torch.from_numpy(np.asarray(padded,dtype=np.int64).copy())


def loss_fn(logits, target):
    valid = target != 255
    if not valid.any():
        return logits.sum()*0
    ce = F.cross_entropy(logits.float(), target, ignore_index=255)
    probabilities = logits.float().softmax(1)*valid[:,None]
    onehot = F.one_hot(target.masked_fill(~valid,0), 5).permute(0,3,1,2).float()*valid[:,None]
    intersection = (probabilities*onehot).sum((0,2,3))
    denominator = (probabilities+onehot).sum((0,2,3))
    # Include all foreground classes; absent-class false positives are penalized.
    dice = 1-((2*intersection[1:]+1)/(denominator[1:]+1)).mean()
    return .5*ce + .5*dice


def evaluate(model, root, rows, size, device, output=None, deadline=None, amp=False):
    model.eval()
    confusion = np.zeros((5,5), dtype=np.int64)
    records = []
    if output:
        output.mkdir(parents=True, exist_ok=True)
    with torch.inference_mode():
        for r in rows:
            if deadline and time.time() >= deadline:
                raise TimeoutError('Evaluation budget exhausted; partial results are not reported as complete')
            with Image.open(root/r['image_file']) as src:
                image = src.convert('RGB')
            with Image.open(root/r['semantic_mask_file']) as src:
                truth = np.asarray(src, dtype=np.int64).copy()
            tensor, (x,y,nw,nh) = prepare_image(image,size)
            with torch.autocast(device_type=device.type, enabled=device.type=='cuda' and amp):
                logits = model(tensor[None].to(device))
            cropped = logits.float()[:,:,y:y+nh,x:x+nw]
            pred = F.interpolate(cropped, size=truth.shape, mode='bilinear', align_corners=False).argmax(1)[0].cpu().numpy()
            valid = truth != 255
            cm = np.bincount(5*truth[valid]+pred[valid], minlength=25).reshape(5,5)
            confusion += cm
            records.append({'image_id':r['image_id'], 'split_group_id':r['split_group_id'],
                            'confusion_matrix':cm.tolist()})
            if output:
                Image.fromarray(pred.astype(np.uint8)).save(output/(r['image_id']+'.png'))
    result = metrics_from_confusion(confusion)
    result['images'] = len(records)
    if output:
        write_json(output.parent/'per_image_metrics.json', records)
    return result


def seed_worker(worker_id):
    if 'torch' not in globals():load_ml()
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)


def worker(args):
    load_ml()
    torch.set_num_threads(2)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device('cpu' if args.cpu else 'cuda:0')
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    root, rows = find_data(args.data_root)
    train = [r for r in rows if r['split']!='test' and int(r['cv_fold'])!=args.worker_fold]
    valid = [r for r in rows if r['split']!='test' and int(r['cv_fold'])==args.worker_fold]
    work = Path(args.work_dir); work.mkdir(parents=True, exist_ok=True)
    model = make_model(pretrained=not args.random_init).to(device)
    encoder = list(model.encoder.parameters())
    enc_ids = {id(p) for p in encoder}
    decoder = [p for p in model.parameters() if id(p) not in enc_ids]
    optimizer = torch.optim.AdamW([{'params':encoder, 'lr':args.lr/10},
                                  {'params':decoder, 'lr':args.lr}], weight_decay=1e-4)
    assert {id(p) for g in optimizer.param_groups for p in g['params']} == {id(p) for p in model.parameters() if p.requires_grad}
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)
    use_amp = device.type == 'cuda' and args.amp
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    generator = torch.Generator().manual_seed(args.seed)
    loader = torch.utils.data.DataLoader(SkinDataset(root,train,args.size,True),
        batch_size=args.batch_size, shuffle=True, num_workers=args.workers,
        pin_memory=device.type=='cuda', worker_init_fn=seed_worker, generator=generator,
        persistent_workers=args.workers>0)
    best, best_epoch, stale = -1., 0, 0
    skipped_nonfinite_loss, skipped_nonfinite_gradient = 0, 0
    start = time.time(); reason = 'epoch_cap'; history=[]; finished_epoch=0
    def save(name, epoch):
        temp = work/(name+'.tmp')
        torch.save({'model':model.state_dict(), 'epoch':epoch, 'best_epoch':best_epoch,
                    'best_dice':best,'optimizer':optimizer.state_dict(),
                    'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                    'args':vars(args),'build':BUILD}, temp)
        temp.replace(work/name)
    for epoch in range(1,args.epochs+1):
        if time.time() >= args.train_deadline:
            reason='time_budget'; break
        model.train()
        # Preserve ImageNet encoder BatchNorm statistics on this small dataset.
        for module in model.encoder.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
        total, batches, timed_out = 0., 0, False
        for inputs, targets in loader:
            if time.time() >= args.train_deadline:
                timed_out=True; break
            optimizer.zero_grad(set_to_none=True)
            inputs=inputs.to(device,non_blocking=True); targets=targets.to(device,non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss=loss_fn(model(inputs),targets)
            if not torch.isfinite(loss):
                skipped_nonfinite_loss += 1
                optimizer.zero_grad(set_to_none=True)
                print(f'fold={args.worker_fold} warning: skipped non-finite loss '
                      f'({skipped_nonfinite_loss}/20)', flush=True)
                if skipped_nonfinite_loss > 20:
                    raise FloatingPointError('More than 20 non-finite losses; training is unstable')
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            if not torch.isfinite(grad_norm):
                skipped_nonfinite_gradient += 1
                optimizer.zero_grad(set_to_none=True)
                if use_amp:
                    scaler.update(max(1., scaler.get_scale()/2))
                print(f'fold={args.worker_fold} warning: skipped non-finite gradient '
                      f'({skipped_nonfinite_gradient}/20)', flush=True)
                if skipped_nonfinite_gradient > 20:
                    raise FloatingPointError('More than 20 non-finite gradients; training is unstable')
                continue
            scaler.step(optimizer); scaler.update()
            total+=loss.item(); batches+=1
        if not batches:
            if timed_out:
                reason='time_budget'; break
            raise FloatingPointError('Epoch completed without any finite optimizer updates')
        score=evaluate(model,root,valid,args.size,device,deadline=args.eval_deadline,amp=use_amp)
        value=score['foreground_macro_dice']
        improved=value>best+1e-4
        if improved:
            best=value; best_epoch=epoch; stale=0
        else:
            stale+=1
        scheduler.step()
        if improved:
            save('best.pth',epoch)
        save('last.pth',epoch)
        finished_epoch=epoch if not timed_out else epoch-1
        history.append({'epoch':epoch,'complete_epoch':not timed_out,'loss':total/batches,
                        'val_dice':value,'val_iou':score['foreground_macro_iou'],
                        'lr':[g['lr'] for g in optimizer.param_groups], 'elapsed_seconds':time.time()-start})
        write_json(work/'history.json',history)
        print(f'fold={args.worker_fold} epoch={epoch}/{args.epochs} loss={total/batches:.4f} '
              f'val Dice={value:.4f} best={best:.4f}',flush=True)
        if timed_out:
            reason='time_budget'; break
        if epoch>=args.min_epochs and stale>=args.patience:
            reason='early_stopping'; break
    if not (work/'best.pth').is_file():
        raise RuntimeError('No trained checkpoint; increase budget or reduce workload')
    checkpoint=torch.load(work/'best.pth',map_location='cpu',weights_only=False)
    model.load_state_dict(checkpoint['model'],strict=True)
    oof=evaluate(model,root,valid,args.size,device,work/'oof_predictions',deadline=args.eval_deadline,amp=use_amp)
    write_json(work/'result.json',{'complete':True,'fold':args.worker_fold,'seed':args.seed,
        'stop_reason':reason,'completed_epochs':finished_epoch,'requested_epochs':args.epochs,
        'best_epoch':best_epoch,'training_and_evaluation_seconds':time.time()-start,'oof':oof,
        'pretrained':not args.random_init,'test_used':False,'amp':use_amp,
        'skipped_nonfinite_loss_batches':skipped_nonfinite_loss,
        'skipped_nonfinite_gradient_batches':skipped_nonfinite_gradient})


def parser():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--data-root'); p.add_argument('--work-dir')
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--epochs',type=int,default=150)
    p.add_argument('--min-epochs',type=int,default=25)
    p.add_argument('--patience',type=int,default=25)
    p.add_argument('--size',type=int,default=384)
    p.add_argument('--batch-size',type=int,default=8)
    p.add_argument('--lr',type=float,default=3e-4)
    p.add_argument('--workers',type=int,default=2)
    p.add_argument('--hours',type=float,default=11.0,help='Overall budget, including evaluation; maximum 11 hours')
    p.add_argument('--reserve-minutes',type=float,default=45)
    p.add_argument('--folds',default='0,1,2,3,4')
    p.add_argument('--audit-only',action='store_true')
    p.add_argument('--amp',action='store_true',help='Optional FP16 AMP; stable research default is FP32')
    p.add_argument('--cpu',action='store_true',help='Local debugging only')
    p.add_argument('--random-init',action='store_true',help='Debugging only; not the pretrained baseline')
    p.add_argument('--worker-fold',type=int,default=None,help=argparse.SUPPRESS)
    p.add_argument('--train-deadline',type=float,default=None,help=argparse.SUPPRESS)
    p.add_argument('--eval-deadline',type=float,default=None,help=argparse.SUPPRESS)
    return p


def main(argv=None):
    args=parser().parse_args(argv)
    if args.worker_fold is not None:
        worker(args); return 0
    root, rows=find_data(args.data_root)
    print(f'{BUILD}: {len(rows)} images; fixed folds; test excluded.',flush=True)
    if args.audit_only:
        print('Manifest/path/group checks passed. No training executed.'); return 0
    folds=[int(x) for x in args.folds.split(',')]
    if not folds or len(set(folds))!=len(folds) or not set(folds)<=set(range(5)):
        raise ValueError('Choose distinct folds from 0,1,2,3,4')
    if not 0<args.hours<=11 or not 0<args.reserve_minutes<args.hours*60:
        raise ValueError('Budget must be <=11 hours with a positive, smaller evaluation reserve')
    if args.epochs<1 or args.batch_size<1 or args.size<64 or args.size%32 or args.workers<0:
        raise ValueError('Positive epochs/batch; image size >=64 divisible by 32; nonnegative workers required')
    session_start=float(os.environ.get('SESSION29_STARTED_AT',time.time()))
    deadline=session_start+args.hours*3600
    load_ml()
    torch.set_num_threads(2)
    if not args.cpu and not torch.cuda.is_available():
        raise RuntimeError('Enable Kaggle GPU T4 x2. --cpu is only for local debugging.')
    gpu_count=1 if args.cpu else min(2,torch.cuda.device_count())
    # Verify the model shape and download official pretrained weights once.
    preflight=make_model(pretrained=not args.random_init)
    with torch.inference_mode():
        preflight.eval()
        assert preflight(torch.zeros(1,3,64,64)).shape==(1,5,64,64)
    del preflight
    # Check every mask before starting expensive jobs; no test model inference.
    for r in rows:
        with Image.open(root/r['semantic_mask_file']) as m:
            if not set(np.unique(m)).issubset({0,1,2,3,4,255}):
                raise ValueError(f'Invalid mask IDs: {r["image_id"]}')
            if m.size != (int(r['width']),int(r['height'])):
                raise ValueError('Mask dimensions disagree with manifest')
    stamp=time.strftime('%Y%m%d_%H%M%S')
    work=Path(args.work_dir or (('/kaggle/working/' if Path('/kaggle/working').is_dir() else './')+f'session29_unet_r34_seed{args.seed}_{stamp}')).resolve()
    if work==root or root in work.parents:
        raise ValueError('Output must be outside the dataset')
    if work.exists() and any(work.iterdir()):
        raise FileExistsError('Use a fresh work directory; existing results are never silently reused')
    work.mkdir(parents=True,exist_ok=True)
    config={'build':BUILD,'args':vars(args),'data_root':str(root),'classes':CLASSES,
        'manifest_sha256':hashlib.sha256((root/'metadata/manifest.csv').read_bytes()).hexdigest(),
        'torch':torch.__version__,'gpu_count':gpu_count,'started_at':session_start,
        'global_deadline':deadline,'test_used':False,'labels':'coarse, clinically unreviewed',
        'metric_warning':'Semantic Dice/IoU, not instance AP; OOF used for checkpoint selection is development evidence.'}
    write_json(work/'run_config.json',config)
    pending=folds.copy(); running={}; failures=[]; done=[]; last_progress=0.
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['OMP_NUM_THREADS']='2'
    try:
        while pending or running:
            for gpu in range(gpu_count):
                if gpu in running or not pending:
                    continue
                available=deadline-time.time()-args.reserve_minutes*60
                if available<=60:
                    failures.extend({'fold':f,'error':'Insufficient remaining time'} for f in pending)
                    pending.clear(); break
                # Divide remaining train time across remaining waves, including running jobs.
                waves=math.ceil((len(pending)+len(running))/gpu_count)
                allocation=available/max(1,waves)
                fold=pending.pop(0); folder=work/f'fold_{fold}'; folder.mkdir()
                command=[sys.executable,str(Path(__file__).resolve()),'--worker-fold',str(fold),
                    '--data-root',str(root),'--work-dir',str(folder),'--seed',str(args.seed),
                    '--epochs',str(args.epochs),'--min-epochs',str(args.min_epochs),'--patience',str(args.patience),
                    '--size',str(args.size),'--batch-size',str(args.batch_size),'--lr',str(args.lr),
                    '--workers',str(args.workers),'--train-deadline',str(time.time()+allocation),
                    '--eval-deadline',str(deadline-120)]
                if args.cpu:command.append('--cpu')
                if args.random_init:command.append('--random-init')
                child_env=env.copy(); child_env['CUDA_VISIBLE_DEVICES']=str(gpu)
                handle=(folder/'train.log').open('w')
                process=subprocess.Popen(command,stdout=handle,stderr=subprocess.STDOUT,env=child_env)
                running[gpu]=(fold,process,handle)
                print(f'Started fold {fold} on GPU slot {gpu}; training allowance {allocation/3600:.2f}h; log {folder}/train.log',flush=True)
            for gpu,(fold,process,handle) in list(running.items()):
                status=process.poll()
                if status is None:continue
                handle.close(); del running[gpu]
                if status==0:
                    done.append(fold); print(f'Fold {fold} complete',flush=True)
                else:
                    failures.append({'fold':fold,'exit_code':status})
                    print((work/f'fold_{fold}/train.log').read_text()[-4000:],flush=True)
            if time.time()-last_progress>=60:
                for fold,process,handle in running.values():
                    lines=(work/f'fold_{fold}/train.log').read_text().splitlines()
                    print(f'[progress fold {fold}] '+(lines[-1] if lines else 'initializing'),flush=True)
                last_progress=time.time()
            if time.time()>=deadline-60:
                failures.extend({'fold':f,'error':'Global deadline'} for f in pending); pending.clear()
                for gpu,(fold,process,handle) in list(running.items()):
                    process.terminate()
                    try:process.wait(timeout=10)
                    except subprocess.TimeoutExpired:process.kill(); process.wait()
                    handle.close(); failures.append({'fold':fold,'error':'Global deadline'})
                    del running[gpu]
                break
            if running:time.sleep(5)
    finally:
        for fold,process,handle in running.values():
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            handle.close()
    total=np.zeros((5,5),dtype=np.int64); fold_results=[]; observed=[]
    for fold in sorted(done):
        result=json.loads((work/f'fold_{fold}/result.json').read_text()); fold_results.append(result)
        total+=np.asarray(result['oof']['confusion_matrix'],dtype=np.int64)
        observed.extend(r['image_id'] for r in json.loads((work/f'fold_{fold}/per_image_metrics.json').read_text()))
    expected=[r['image_id'] for r in rows if r['split']!='test']
    complete=not failures and set(folds)==set(range(5)) and sorted(observed)==sorted(expected)
    summary={'complete_five_fold_oof':complete,'failures':failures,'fold_results':fold_results,
             'test_used':False,'seed':args.seed,'pretrained':not args.random_init,
             'elapsed_hours':(time.time()-session_start)/3600,
             'pooled_oof':metrics_from_confusion(total) if done else None,
             'fold_dice_mean':float(np.mean([r['oof']['foreground_macro_dice'] for r in fold_results])) if done else None,
             'fold_dice_std':float(np.std([r['oof']['foreground_macro_dice'] for r in fold_results],ddof=1)) if len(done)>1 else None,
             'limitation':'Development OOF with validation-selected checkpoints; coarse masks; no external validation. One seed does not estimate seed variability.'}
    write_json(work/'summary.json',summary)
    print(json.dumps(summary,indent=2),flush=True)
    print('RESULTS:',work,flush=True)
    return 1 if failures else 0


if __name__=='__main__':
    # A pasted Kaggle cell lacks __file__. Materialize that cell once, then execute
    # it as a real script because the fold launcher must have a filesystem path.
    if '__file__' not in globals():
        notebook_source = globals().get('In', [])[-1] if globals().get('In') else ''
        if not notebook_source or 'SESSION29-UNET-R34-V1' not in notebook_source:
            raise RuntimeError('Use SESSION_29_KAGGLE_RUN_THIS.ipynb, or run the saved .py file with Python.')
        runtime_dir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
        runtime = runtime_dir/'SESSION_29_runtime_from_cell.py'
        runtime.write_text(notebook_source, encoding='utf-8')
        completed = subprocess.run([sys.executable, '-u', str(runtime)], text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(completed.stdout, end='', flush=True)
        if completed.returncode:
            tail = completed.stdout[-8000:] if completed.stdout else 'No child-process output was captured.'
            raise RuntimeError(f'Session 29 runtime failed with exit status {completed.returncode}.\n'
                               f'Last runtime output:\n{tail}')
    else:
        raise SystemExit(main())


SESSION29-UNET-R34-V1: 268 images; fixed folds; test excluded.
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth

  0%|          | 0.00/83.3M [00:00<?, ?B/s]
 15%|█▌        | 12.5M/83.3M [00:00<00:00, 130MB/s]
 36%|███▌      | 30.0M/83.3M [00:00<00:00, 161MB/s]
 57%|█████▋    | 47.4M/83.3M [00:00<00:00, 170MB/s]
 78%|███████▊  | 64.8M/83.3M [00:00<00:00, 175MB/s]
100%|██████████| 83.3M/83.3M [00:00<00:00, 174MB/s]
Started fold 0 on GPU slot 0; training allowance 3.42h; log /kaggle/working/session29_unet_r34_seed42_20260922_112356/fold_0/train.log
Started fold 1 on GPU slot 1; training allowance 3.42h; log /kaggle/working/session29_unet_r34_seed42_20260922_112356/fold_1/train.log
[progress fold 0] initializing
[progress fold 1] initializing
[progress fold 0] fold=0 epoch=5/150 loss=0.9690 val Dice=0.2001 best=0.2563
[progress fold 1] fold=1 epoch=5/150 loss=0.9530 val Dice=0.2726 best=0.3045
[progress fol